<a href="https://colab.research.google.com/github/shreyansh8h/hello-world/blob/master/May30_ArulA2Acommunictaion_ADK.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Agent-to-Agent (A2A) Communication with Google ADK and OpenAI GPT-4o Mini

This Colab notebook demonstrates Google ADK's A2A pattern with an OpenAI-backed model.

You will build two agents:

1. **Product Catalog Agent**: exposed as a remote A2A service.
2. **Customer Support Agent**: consumes the remote catalog agent through `RemoteA2aAgent`.

Both agents use `gpt-4o-mini` through ADK's LiteLLM integration, and the OpenAI key is read from Google Colab Secrets.

In [ ]:
# Install Google ADK with A2A support plus the OpenAI/LiteLLM dependencies.
# Run this once per fresh Colab runtime, then restart the runtime if Colab asks you to.
%pip install -q "google-adk[a2a]" litellm openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.0/17.0 MB 47.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 151.0/151.0 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 10.3 MB/s eta 0:00:00


## Architecture

```text
+----------------------+          A2A HTTP           +----------------------+
| Customer Support     |  ------------------------>  | Product Catalog      |
| Agent (consumer)     |                             | Agent (remote)       |
| OpenAI gpt-4o-mini   |                             | OpenAI gpt-4o-mini   |
+----------------------+                             +----------------------+
```

The catalog agent is treated like an external vendor-owned service. The support agent does not call the catalog tool directly; it talks to the published A2A endpoint and uses the remote agent card as the contract.

## Colab Secret Setup

In Colab, open **Secrets** from the left sidebar and add:

- Name: `OPENAI_API_KEY`
- Value: your OpenAI API key

Enable notebook access for that secret before running the next cell.

In [ ]:
import json
import os
import requests
import subprocess
import time
import uuid
import warnings

from google.adk.agents import LlmAgent
from google.adk.agents.remote_a2a_agent import (
    AGENT_CARD_WELL_KNOWN_PATH,
    RemoteA2aAgent,
)
from google.adk.a2a.utils.agent_to_a2a import to_a2a
from google.adk.models.lite_llm import LiteLlm
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

warnings.filterwarnings("ignore")

OPENAI_MODEL = "openai/gpt-4o-mini"
A2A_HOST = "localhost"
A2A_PORT = 8001
A2A_BASE_URL = f"http://{A2A_HOST}:{A2A_PORT}"

print("ADK, A2A, and LiteLLM imports completed.")
print(f"Model configured for both agents: {OPENAI_MODEL}")

/usr/local/lib/python3.12/dist-packages/google/adk/features/_feature_decorator.py:72: UserWarning: [EXPERIMENTAL] feature FeatureName.PLUGGABLE_AUTH is enabled.
  check_feature_enabled()


ADK, A2A, and LiteLLM imports completed.
Model configured for both agents: openai/gpt-4o-mini


In [ ]:
def load_colab_secret(secret_name: str) -> str:
    """Load a secret from environment first, then Google Colab Secrets."""
    value = os.environ.get(secret_name)
    if value:
        return value

    try:
        from google.colab import userdata
    except ImportError as exc:
        raise RuntimeError(
            f"{secret_name} is not set. In Colab, add it under Secrets; "
            "outside Colab, set it as an environment variable."
        ) from exc

    value = userdata.get(secret_name)
    if not value:
        raise RuntimeError(
            f"{secret_name} was not found in Colab Secrets. "
            "Create it in the Secrets panel and enable notebook access."
        )

    os.environ[secret_name] = value
    return value


OPENAI_API_KEY = load_colab_secret("OPENAI_API_KEY")

# LiteLLM and the OpenAI SDK both read this environment variable.
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

print("OPENAI_API_KEY loaded from Colab Secrets/environment.")

OPENAI_API_KEY loaded from Colab Secrets/environment.


## Product Catalog Tool

This function simulates a vendor inventory system. The remote catalog agent will use it as a tool when product questions arrive over A2A.

In [ ]:
def fetch_product_details(item_name: str) -> str:
    """Return catalog details for a requested product."""
    inventory_snapshot = {
        "iphone 15 pro": "iPhone 15 Pro, $999, Low Stock (8 units), 128GB, Titanium finish",
        "samsung galaxy s24": "Samsung Galaxy S24, $799, In Stock (31 units), 256GB, Phantom Black",
        "dell xps 15": "Dell XPS 15, $1,299, In Stock (45 units), 15.6-inch display, 16GB RAM, 512GB SSD",
        "macbook pro 14": "MacBook Pro 14-inch, $1,999, In Stock (22 units), M3 Pro chip, 18GB RAM, 512GB SSD",
        "sony wh-1000xm5": "Sony WH-1000XM5 Headphones, $399, In Stock (67 units), Noise-canceling, 30-hour battery",
        "ipad air": "iPad Air, $599, In Stock (28 units), 10.9-inch display, 64GB",
        "lg ultrawide 34": "LG UltraWide 34-inch Monitor, $499, Out of Stock, Expected: Next week",
    }

    key = item_name.strip().lower()
    if key in inventory_snapshot:
        return f"Product Details: {inventory_snapshot[key]}"

    supported_items = ", ".join(name.title() for name in inventory_snapshot)
    return (
        f"Details for '{item_name}' are unavailable. "
        f"Supported products include: {supported_items}."
    )


print(fetch_product_details("iPhone 15 Pro"))

Product Details: iPhone 15 Pro, $999, Low Stock (8 units), 128GB, Titanium finish


## Define the Remote Catalog Agent

The same agent definition is used in the notebook for clarity and written to `/tmp/catalog_agent_server.py` so `uvicorn` can serve it as an A2A app.

In [ ]:
catalog_lookup_agent = LlmAgent(
    model=LiteLlm(model=OPENAI_MODEL),
    name="catalog_lookup_agent",
    description=(
        "External vendor product catalog agent that shares product "
        "specifications, pricing, and stock status."
    ),
    instruction="""
    You represent an external vendor's product information desk.
    For any product-related inquiry, use the fetch_product_details tool.
    Clearly include price, stock status, and key specifications.
    If multiple products are mentioned, address each one separately.
    Keep responses concise, professional, and customer-friendly.
    """,
    tools=[fetch_product_details],
)

catalog_agent_service_preview = to_a2a(catalog_lookup_agent, port=A2A_PORT)

print("Catalog Lookup Agent defined with OpenAI gpt-4o-mini.")
print("A2A app can be generated from this agent with to_a2a(...).")

Catalog Lookup Agent defined with OpenAI gpt-4o-mini.
A2A app can be generated from this agent with to_a2a(...).


In [ ]:
catalog_agent_source = f"""
import os

from google.adk.agents import LlmAgent
from google.adk.a2a.utils.agent_to_a2a import to_a2a
from google.adk.models.lite_llm import LiteLlm

OPENAI_MODEL = "{OPENAI_MODEL}"

if not os.environ.get("OPENAI_API_KEY"):
    raise RuntimeError(
        "OPENAI_API_KEY is not set. The notebook must load it before starting uvicorn."
    )

def fetch_product_details(item_name: str) -> str:
    inventory_snapshot = {{
        "iphone 15 pro": "iPhone 15 Pro, $999, Low Stock (8 units), 128GB, Titanium finish",
        "samsung galaxy s24": "Samsung Galaxy S24, $799, In Stock (31 units), 256GB, Phantom Black",
        "dell xps 15": "Dell XPS 15, $1,299, In Stock (45 units), 15.6-inch display, 16GB RAM, 512GB SSD",
        "macbook pro 14": "MacBook Pro 14-inch, $1,999, In Stock (22 units), M3 Pro chip, 18GB RAM, 512GB SSD",
        "sony wh-1000xm5": "Sony WH-1000XM5 Headphones, $399, In Stock (67 units), Noise-canceling, 30-hour battery",
        "ipad air": "iPad Air, $599, In Stock (28 units), 10.9-inch display, 64GB",
        "lg ultrawide 34": "LG UltraWide 34-inch Monitor, $499, Out of Stock, Expected: Next week",
    }}

    key = item_name.strip().lower()
    if key in inventory_snapshot:
        return f"Product Details: {{inventory_snapshot[key]}}"

    supported_items = ", ".join(name.title() for name in inventory_snapshot)
    return (
        f"Details for '{{item_name}}' are unavailable. "
        f"Supported products include: {{supported_items}}."
    )

catalog_agent = LlmAgent(
    model=LiteLlm(model=OPENAI_MODEL),
    name="catalog_lookup_agent",
    description=(
        "External vendor product catalog agent that shares product "
        "specifications, pricing, and stock status."
    ),
    instruction=(
        "You represent an external vendor's product information desk. "
        "For any product-related inquiry, use the fetch_product_details tool. "
        "Clearly include price, stock status, and key specifications. "
        "If multiple products are mentioned, address each one separately. "
        "Keep responses concise, professional, and customer-friendly."
    ),
    tools=[fetch_product_details],
)

app = to_a2a(catalog_agent, port={A2A_PORT})
"""

agent_file_path = "/tmp/catalog_agent_server.py"
with open(agent_file_path, "w", encoding="utf-8") as file_handle:
    file_handle.write(catalog_agent_source)

print(f"Catalog agent server module written to {agent_file_path}")

Catalog agent server module written to /tmp/catalog_agent_server.py


## Start the A2A Server

This cell starts a local `uvicorn` process and waits until the agent card is reachable.

In [ ]:
# Stop an earlier server if this notebook cell is rerun.
existing_process = globals().get("catalog_agent_server_process")
if existing_process and existing_process.poll() is None:
    existing_process.terminate()
    existing_process.wait(timeout=10)
    print("Stopped previous Catalog Lookup Agent server.")

server_handle = subprocess.Popen(
    [
        "uvicorn",
        "catalog_agent_server:app",
        "--host",
        A2A_HOST,
        "--port",
        str(A2A_PORT),
    ],
    cwd="/tmp",
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    env={**os.environ},
    text=True,
)

globals()["catalog_agent_server_process"] = server_handle

print("Launching Catalog Lookup Agent A2A server...")

max_checks = 30
for check in range(max_checks):
    try:
        result = requests.get(f"{A2A_BASE_URL}{AGENT_CARD_WELL_KNOWN_PATH}", timeout=1)
        if result.status_code == 200:
            print("Catalog Lookup Agent server is live.")
            print(f"Base URL: {A2A_BASE_URL}")
            print(f"Agent Card: {A2A_BASE_URL}{AGENT_CARD_WELL_KNOWN_PATH}")
            break
    except requests.exceptions.RequestException:
        pass

    if server_handle.poll() is not None:
        _, stderr_text = server_handle.communicate(timeout=2)
        raise RuntimeError(f"A2A server exited during startup:\n{stderr_text}")

    time.sleep(2)
    print(".", end="", flush=True)
else:
    raise TimeoutError("A2A server did not publish an agent card in time.")

Launching Catalog Lookup Agent A2A server...
.....Catalog Lookup Agent server is live.
Base URL: http://localhost:8001
Agent Card: http://localhost:8001/.well-known/agent-card.json


In [ ]:
card_response = requests.get(f"{A2A_BASE_URL}{AGENT_CARD_WELL_KNOWN_PATH}", timeout=5)
card_response.raise_for_status()

catalog_agent_card = card_response.json()
print("Catalog Lookup Agent Card")
print(json.dumps(catalog_agent_card, indent=2))

print("\nAgent overview")
print(f"Name: {catalog_agent_card.get('name')}")
print(f"Description: {catalog_agent_card.get('description')}")
print(f"Endpoint: {catalog_agent_card.get('url')}")
print(f"Skills exposed: {len(catalog_agent_card.get('skills', []))}")

Catalog Lookup Agent Card
{
  "capabilities": {},
  "defaultInputModes": [
    "text/plain"
  ],
  "defaultOutputModes": [
    "text/plain"
  ],
  "description": "External vendor product catalog agent that shares product specifications, pricing, and stock status.",
  "name": "catalog_lookup_agent",
  "preferredTransport": "JSONRPC",
  "protocolVersion": "0.3.0",
  "skills": [
    {
      "description": "External vendor product catalog agent that shares product specifications, pricing, and stock status. I represent an external vendor's product information desk. For any product-related inquiry, use the fetch_product_details tool. Clearly include price, stock status, and key specifications. If multiple products are mentioned, address each one separately. Keep responses concise, professional, and customer-friendly.",
      "examples": [],
      "id": "catalog_lookup_agent",
      "name": "model",
      "tags": [
        "llm"
      ]
    },
    {
      "description": "Call self as a functi

## Consume the Remote Agent

`RemoteA2aAgent` reads the agent card and exposes the remote catalog service as a sub-agent for the local customer support agent.

In [ ]:
catalog_agent_proxy = RemoteA2aAgent(
    name="catalog_lookup_agent",
    description=(
        "Client proxy for the external vendor catalog agent exposed over A2A."
    ),
    agent_card=f"{A2A_BASE_URL}{AGENT_CARD_WELL_KNOWN_PATH}",
)

support_agent = LlmAgent(
    model=LiteLlm(model=OPENAI_MODEL),
    name="customer_support_agent",
    description=(
        "Customer support assistant that answers product inquiries by "
        "delegating catalog lookups to a remote A2A agent."
    ),
    instruction="""
    You are a professional and approachable customer support agent.

    For any product-related question:
    1. Use the catalog_lookup_agent sub-agent to fetch product details.
    2. Answer with price, stock status, and key specifications.
    3. Mention restock timing when the product is unavailable.
    4. Keep the final answer concise and helpful.

    Always retrieve product data from catalog_lookup_agent before answering.
    """,
    sub_agents=[catalog_agent_proxy],
)

print("Customer Support Agent initialized.")
print(f"Model in use: {OPENAI_MODEL}")
print(f"Remote sub-agents: {len(support_agent.sub_agents)}")

Customer Support Agent initialized.
Model in use: openai/gpt-4o-mini
Remote sub-agents: 1


## Test End-to-End A2A Communication

Each query starts a new session, sends a user request to the support agent, and prints the final answer after the support agent collaborates with the remote catalog agent.

In [ ]:
async def test_customer_support_a2a(query: str) -> None:
    """Run one customer query through the support agent and remote catalog agent."""
    session_service = InMemorySessionService()
    app_name = "support_app"
    user_id = "demo_user"
    session_id = f"session_{uuid.uuid4().hex[:8]}"

    await session_service.create_session(
        app_name=app_name,
        user_id=user_id,
        session_id=session_id,
    )

    runner = Runner(
        agent=support_agent,
        app_name=app_name,
        session_service=session_service,
    )

    message = types.Content(role="user", parts=[types.Part(text=query)])

    print(f"\nUser Query: {query}")
    print("Agent Response:")
    print("=" * 72)

    final_response_seen = False
    async for event in runner.run_async(
        user_id=user_id,
        session_id=session_id,
        new_message=message,
    ):
        if event.is_final_response() and event.content:
            final_response_seen = True
            for part in event.content.parts:
                if getattr(part, "text", None):
                    print(part.text)

    if not final_response_seen:
        print("No final response was returned by the agent.")

    print("=" * 72)

In [ ]:
await test_customer_support_a2a(
    "Can you provide details on the iPhone 15 Pro? Is it currently in stock?"
)


User Query: Can you provide details on the iPhone 15 Pro? Is it currently in stock?
Agent Response:


13:00:39 - LiteLLM:WARNING: common_utils.py:979 - litellm: could not pre-load bedrock-runtime response stream shape — Bedrock event-stream decoding will be unavailable. Error: No module named 'botocore'
13:00:39 - LiteLLM:WARNING: common_utils.py:24 - litellm: could not pre-load sagemaker-runtime response stream shape — SageMaker event-stream decoding will be unavailable. Error: No module named 'botocore'
INFO:google_adk.google.adk.agents.remote_a2a_agent:Successfully resolved remote A2A agent: catalog_lookup_agent


The iPhone 15 Pro is priced at **$999**. It currently has **low stock** with **8 units** available. Key specifications include **128GB of storage** and a **titanium finish**. If you have any further questions or need assistance, feel free to ask!


In [ ]:
await test_customer_support_a2a(
    "I'm looking for a laptop. Can you compare the Dell XPS 15 and MacBook Pro 14 for me?"
)


User Query: I'm looking for a laptop. Can you compare the Dell XPS 15 and MacBook Pro 14 for me?
Agent Response:
Here’s a comparison of the Dell XPS 15 and MacBook Pro 14:

### Dell XPS 15
- **Price:** $1,299 
- **Stock Status:** In Stock (45 units)
- **Key Specifications:**
  - Display: 15.6 inches 
  - RAM: 16GB 
  - Storage: 512GB SSD 

### MacBook Pro 14
- **Price:** $1,999 
- **Stock Status:** In Stock (22 units)
- **Key Specifications:**
  - Chip: M3 Pro 
  - RAM: 18GB 
  - Storage: 512GB SSD 

If you need more details or have further questions, feel free to ask!


In [ ]:
await test_customer_support_a2a(
    "Do you have the Sony WH-1000XM5 headphones? What's the price?"
)


User Query: Do you have the Sony WH-1000XM5 headphones? What's the price?
Agent Response:
The **Sony WH-1000XM5 headphones** are available for **$399**. There are **67 units in stock**. 

**Key Specifications:**
- Type: Noise-canceling
- Battery Life: 30 hours

If you have any further questions or need assistance, feel free to ask!


## Cleanup

Run this cell when you are done testing or before restarting the A2A server on the same port.

In [ ]:
server_process = globals().get("catalog_agent_server_process")
if server_process and server_process.poll() is None:
    server_process.terminate()
    server_process.wait(timeout=10)
    print("Catalog Lookup Agent server stopped.")
else:
    print("No running Catalog Lookup Agent server found.")

Catalog Lookup Agent server stopped.


## Further Reading

- Google ADK A2A overview: https://google.github.io/adk-docs/a2a/intro/
- Expose an ADK agent with A2A: https://google.github.io/adk-docs/a2a/quickstart-exposing/
- Consume a remote A2A agent: https://google.github.io/adk-docs/a2a/quickstart-consuming/
- OpenAI model reference: https://platform.openai.com/docs/models